In [ ]:
# @title 📊 Kalshi SOL Market Insights Dashboard {display-mode: "form"}

import pandas as pd
import json
from IPython.display import HTML
from google.colab import output

# --- Backend Data Processing ---
def _report_js_error(message):
    print(f"JavaScript Error: {message}")

output.register_callback('report_js_error', _report_js_error)

# Load results from the specified CSV file (SOL)
df_markets = pd.read_csv('/content/SOL.csv')

# Prepare data for JS
dashboard_df = df_markets.copy()
dashboard_df['open_time'] = pd.to_datetime(dashboard_df['open_time'])
dashboard_df['open_time_str'] = dashboard_df['open_time'].dt.strftime('%Y-%m-%d %H:%M')

# Calculate KPIs
total_markets = len(dashboard_df)
yes_count = len(dashboard_df[dashboard_df['result'].str.lower() == 'yes'])
yes_rate = (yes_count / total_markets * 100) if total_markets > 0 else 0
avg_expiration = pd.to_numeric(dashboard_df['expiration_value'], errors='coerce').mean()

kpi_data = {
    "total": total_markets,
    "yes_rate": f"{yes_rate:.1f}%",
    "avg_exp": f"${avg_expiration:,.2f}"
}

json_data = dashboard_df.to_json(orient='records')

# --- Dashboard Template ---
html_template = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap" rel="stylesheet">
    <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
    <style>
        :root {
            --bg-color: #f4f6f8;
            --card-bg: #ffffff;
            --primary: #2563eb;
            --text-main: #1e293b;
            --text-muted: #64748b;
            --shadow: 0 4px 6px -1px rgb(0 0 0 / 0.1), 0 2px 4px -2px rgb(0 0 0 / 0.1);
        }
        body {
            font-family: 'Inter', sans-serif;
            background-color: var(--bg-color);
            color: var(--text-main);
            margin: 0;
            padding: 20px;
        }
        .dashboard-container {
            max-width: 1200px;
            margin: 0 auto;
            display: flex; flex-direction: column; gap: 24px;
        }
        .kpi-row {
            display: grid; grid-template-columns: repeat(auto-fit, minmax(240px, 1fr)); gap: 20px;
        }
        .card {
            background: var(--card-bg); padding: 24px; border-radius: 12px; box-shadow: var(--shadow);
        }
        .kpi-card { text-align: center; }
        .kpi-value { font-size: 2rem; font-weight: 700; color: var(--primary); margin: 8px 0; }
        .kpi-label { color: var(--text-muted); font-size: 0.875rem; text-transform: uppercase; letter-spacing: 0.05em; }
        .charts-row {
            display: grid; grid-template-columns: 2fr 1fr; gap: 20px;
        }
        .canvas-wrapper {
            position: relative; flex-grow: 1; min-height: 300px;
        }
        .table-container {
            overflow-x: auto; background: white; border-radius: 12px; box-shadow: var(--shadow);
        }
        table {
            width: 100%; border-collapse: collapse; font-size: 0.875rem;
        }
        th {
            background: #f8fafc; text-align: left; padding: 12px 16px; color: var(--text-muted); font-weight: 600; border-bottom: 1px solid #e2e8f0;
        }
        td { padding: 12px 16px; border-bottom: 1px solid #f1f5f9; }
        .badge { padding: 4px 8px; border-radius: 4px; font-weight: 600; font-size: 0.75rem; text-transform: uppercase; }
        .badge-yes { background: #dcfce7; color: #166534; }
        .badge-no { background: #fee2e2; color: #991b1b; }
        @media (max-width: 768px) {
            .charts-row { grid-template-columns: 1fr; }
        }
    </style>
</head>
<body>
    <div class="dashboard-container">
        <h2 style="margin:0;">SOL Market Analysis Dashboard</h2>
        <div class="kpi-row">
            <div class="card kpi-card">
                <div class="kpi-label">Total Markets</div>
                <div class="kpi-value" id="kpi-total">0</div>
            </div>
            <div class="card kpi-card">
                <div class="kpi-label">YES Outcome Rate</div>
                <div class="kpi-value" id="kpi-rate">0%</div>
            </div>
            <div class="card kpi-card">
                <div class="kpi-label">Avg Expiration ($)</div>
                <div class="kpi-value" id="kpi-exp">$0</div>
            </div>
        </div>

        <div class="charts-row">
            <div class="card" style="display:flex; flex-direction:column;">
                <h3 style="margin-top:0;">SOL Expiration Values</h3>
                <div class="canvas-wrapper">
                    <canvas id="mainLineChart"></canvas>
                </div>
            </div>
            <div class="card" style="display:flex; flex-direction:column;">
                <h3 style="margin-top:0;">Outcome Split</h3>
                <div class="canvas-wrapper">
                    <canvas id="outcomeDonutChart"></canvas>
                </div>
            </div>
        </div>

        <div class="table-container">
            <table>
                <thead>
                    <tr>
                        <th>Ticker</th>
                        <th>Open Time (EST)</th>
                        <th>Strike</th>
                        <th>Exp Value</th>
                        <th>Result</th>
                    </tr>
                </thead>
                <tbody id="tableBody"></tbody>
            </table>
        </div>
    </div>

    <script>
        window.onerror = function(message) {
            google.colab.kernel.invokeFunction('report_js_error', [message], {});
        };

        const data = DATA_PLACEHOLDER;
        const kpis = KPI_PLACEHOLDER;

        function initDashboard() {
            document.getElementById('kpi-total').innerText = kpis.total;
            document.getElementById('kpi-rate').innerText = kpis.yes_rate;
            document.getElementById('kpi-exp').innerText = kpis.avg_exp;

            const sortedData = [...data].sort((a, b) => a.open_time - b.open_time);

            new Chart(document.getElementById('mainLineChart'), {
                type: 'line',
                data: {
                    labels: sortedData.map(d => d.open_time_str),
                    datasets: [{
                        label: 'SOL Expiration',
                        data: sortedData.map(d => d.expiration_value),
                        borderColor: '#8b5cf6',
                        backgroundColor: 'rgba(139, 92, 246, 0.1)',
                        fill: true,
                        tension: 0.3,
                        pointRadius: 1
                    }]
                },
                options: {
                    responsive: true,
                    maintainAspectRatio: false,
                    plugins: { legend: { display: false } },
                    scales: {
                        x: { display: false },
                        y: { beginAtZero: false }
                    }
                }
            });

            const outcomes = data.reduce((acc, curr) => {
                const r = curr.result.toLowerCase();
                acc[r] = (acc[r] || 0) + 1;
                return acc;
            }, {});

            new Chart(document.getElementById('outcomeDonutChart'), {
                type: 'doughnut',
                data: {
                    labels: ['Yes', 'No'],
                    datasets: [{
                        data: [outcomes.yes || 0, outcomes.no || 0],
                        backgroundColor: ['#22c55e', '#ef4444']
                    }]
                },
                options: {
                    responsive: true,
                    maintainAspectRatio: false,
                    plugins: { legend: { position: 'bottom' } }
                }
            });

            const tableBody = document.getElementById('tableBody');
            data.slice(0, 50).forEach(row => {
                const tr = document.createElement('tr');
                tr.innerHTML = `
                    <td>${row.event_ticker}</td>
                    <td>${row.open_time_str}</td>
                    <td>${row.floor_strike}</td>
                    <td>${row.expiration_value}</td>
                    <td><span class="badge badge-${row.result.toLowerCase()}">${row.result}</span></td>
                `;
                tableBody.appendChild(tr);
            });
        }

        initDashboard();
    </script>
</body>
</html>
"""

# Inject data into template using string replacement
final_html = html_template.replace('DATA_PLACEHOLDER', json_data)
final_html = final_html.replace('KPI_PLACEHOLDER', json.dumps(kpi_data))

display(HTML(final_html))
